<a href="https://colab.research.google.com/github/tahumada/MSO-AEON/blob/main/DECam/DECam_submit_from_json2AEON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DECam target submission to the AEON queue
Tomas Ahumada - tomas.ahumada@noirlab.edu

Last modified: Aug 2026

This code intends to be a tutrorial for NOIRLab-AEON users of the Dark Energy Camera (DECam) mounted at the 4m V. M. Blanco telescope. The AEON queue is run by the Las Cumbres Observatory (LCO) Scheduler, thus the user requires an active account to access the LCO portal and generate a LCO key to submit requests to an active program.

Particularly, this notebook intends to serve as a translator between the json files submitted directly to SISPI and AEON.

Information about DECam can be found here: https://noirlab.edu/science/programs/ctio/instruments/Dark-Energy-Camera
Information about LCO can be found here: https://observe.lco.global/

Once you have an active user in the LCO portal, you can find the API key here https://observe.lco.global/accounts/profile

# Outline
1. Read example target list from a json file. the json file should have the format that is SISPI compatible.
2. Get template request (json format) - this json file is modified and later sent to the LCO queue
3. Make the payload, modifying the json template.
4. Submit targets individually. Each target can have multiple filters.

In [1]:
import copy
from datetime import datetime, timedelta
import glob
import io
import json
import os
import re
import time
import urllib.request
from urllib.parse import urlparse
from astropy.time import Time
import numpy as np
import pandas as pd
import requests
from pathlib import Path


# Types of JSON files that this code manages:

SISPI JSON files require either absolute coordinates (RA, Dec) or relative offsets (offset_ra, offset_dec). A single target cannot have both. Because offsets are calculated relative to the previous position, this code supports JSON files formatted in one of two ways:

* All targets use absolute coordinates.

* The first target uses absolute coordinates, and all subsequent targets use offsets.

In [2]:
# Case A: all targets have RA and Dec
url_json = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/DECam/example_json_coordinates.json"

targets_coord = pd.read_json(url_json)

targets_coord

,expTime,RA,dec,count,expType,program,wait,propid,proposer,comment,object,filter
0,30,78.907229,-21.416664,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_g6,g
1,30,78.907229,-21.416664,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_r6,r
2,30,141.163996,-8.008000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_g9,g
3,30,141.163996,-8.008000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_r9,r
4,30,132.665000,0.000000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_g2,g
5,30,132.665000,0.000000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_r2,r
6,30,137.221996,1.526000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_g8,g
7,30,137.221996,1.526000,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_r8,r
8,30,149.224167,1.748056,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_g5,g
9,30,149.224167,1.748056,1,object,ZTF+DECam,False,2025B-240381,Ahumada,ZTF+DECam,pointing_r5,r


In [3]:
# get template AEON json
def get_example(example_path):
    if "github.com" in example_path and "/blob/" in example_path:
        example_path = example_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    if urlparse(example_path).scheme in ('http', 'https'):
        response = requests.get(example_path)

        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Failed to fetch file from URL. Status code: {response.status_code}")
    else:
        pass



In [4]:
template_json_url = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/DECam/example.json"
print('The template of an observation:')
get_example(template_json_url)

The template of an observation:


{'id': 2495241,
 'requests': [{'id': 4168585,
   'location': {'telescope_class': '4m0'},
   'configurations': [{'id': 14811742,
     'constraints': {'max_airmass': 1.6,
      'min_lunar_distance': 30.0,
      'max_lunar_phase': 1.0,
      'max_seeing': None,
      'min_transparency': None,
      'extra_params': {}},
     'instrument_configs': [{'exposure_time': 40.0,
       'optical_elements': {'filter': 'r'},
       'mode': 'default',
       'exposure_count': 1,
       'rotator_mode': '',
       'extra_params': {'offset_ra': 0, 'offset_dec': 0}}],
     'acquisition_config': {'mode': 'OFF',
      'exposure_time': None,
      'extra_params': {}},
     'guiding_config': {'optional': True,
      'mode': 'ON',
      'optical_elements': {},
      'exposure_time': None,
      'extra_params': {}},
     'target': {'type': 'ICRS',
      'name': '',
      'ra': 151.197,
      'dec': 7.801,
      'proper_motion_ra': 0.0,
      'proper_motion_dec': 0.0,
      'parallax': 0.0,
      'epoch': 2000.0

In [5]:
def make_targetlist(json_path):

  df_loaded_json = pd.read_json(json_path)

  target_list = []

  # Group the dataframe by RA and Dec
  grouped = df_loaded_json.groupby(['RA', 'dec'])

  for (ra, dec), group_data in grouped:
      # Assuming the target name is the same for all rows in this RA/Dec group,
      # we just grab the name from the very first row of the group.
      name = group_data['object'].iloc[0]

      # Extract the filters and exposure times into lists
      filter_list = group_data['filter'].tolist()
      exptime_key =  'exptime' if 'exptime' in df_loaded_json.columns else 'expTime'
      exptime_list = group_data[exptime_key].tolist()

      if 'count' in group_data.keys():
        count_list = group_data['count'].tolist()
      else:
        count_list = np.ones(len(filter_list)).tolist()

      # Create the dictionary for this specific location
      target_dict = {
          'id': name,
          'ra': ra,
          'dec': dec,
          'counts': count_list,
          'filters': filter_list,
          'exptime': exptime_list
      }

      # Append to our main list
      target_list.append(target_dict)


  # json_file_name = Path(json_path).stem
  target_list.append({'json_file_name': json_path})

  if len(np.unique(df_loaded_json['propid']))!= 1:
    raise Exception("more than one propID")
  target_list.append({'propid': np.unique(df_loaded_json['propid'])[0]})

  return target_list


In [6]:
import copy
from astropy.time import Time

def make_payload(variables, template_path,decat=True,break_expcount=False):
    data = get_example(template_path)

    # We take the first configuration and its first instrument config to copy later
    base_req_config = data['requests'][0]['configurations'][0]
    base_instrument_config = base_req_config['instrument_configs'][0]

    # Clear configurations
    data['requests'][0]['configurations'] = []


    # Set top-level request variables
    # Since there are multiple targets, we'll name the payload using the first target's ID
    data['name'] = variables['json_file_name']
    data['proposal'] = variables['PROPOSAL_ID']
    data['requests'][0]['windows'] = variables['windows']
    data['ipp_value'] = variables['ipp_value']


    # Loop through each target provided in the variables dictionary
    for target_data in variables['targets']:

        # Create a completely independent copy of the base configuration blueprint
        new_config = copy.deepcopy(base_req_config)

        # Set Constraints
        new_config['constraints']['max_airmass'] = variables['maximum_airmass']
        new_config['constraints']['min_lunar_distance'] = variables['minimum_lunar_distance']

        # Set Target Info (RA, Dec, Name)
        new_config['target']['name'] = target_data['id']
        new_config['target']['ra'] = str(target_data['ra'])
        new_config['target']['dec'] = str(target_data['dec'])

        # Centering
        if 'extra_params' not in new_config:
            new_config['extra_params'] = {}
        if decat:
            new_config['extra_params']['detector_centering'] = 'central_gap'
        else:
            new_config['extra_params']['detector_centering'] = 'S4'

        # Clear the instrument configs in this specific configuration copy
        new_config['instrument_configs'] = []

        # 3. Loop through filters for THIS specific target
        if break_expcount == False:
          for filt in ['g', 'r', 'i', 'z']:
              if filt in target_data['filters']:
                  # Create an independent copy of the instrument config blueprint
                  new_inst_config = copy.deepcopy(base_instrument_config)
                  new_inst_config['optical_elements']['filter'] = filt
                  idx = list(target_data['filters']).index(filt)
                  new_inst_config['exposure_time'] = str(float(target_data['exptime'][idx]))
                  new_inst_config['exposure_count']= str(float(target_data['counts'][idx]))
                  new_config['instrument_configs'].append(new_inst_config)
          # 4. Append this completed target configuration back to the main payload array
          data['requests'][0]['configurations'].append(new_config)

        elif break_expcount:
          for filt in ['g', 'r', 'i', 'z']:
                if filt in target_data['filters']:
                  idx = list(target_data['filters']).index(filt)
                  for i in range(target_data['counts'][idx]):
                    # Create an independent copy of the instrument config blueprint
                    new_inst_config = copy.deepcopy(base_instrument_config)
                    new_inst_config['optical_elements']['filter'] = filt
                    new_inst_config['exposure_time'] = str(float(target_data['exptime'][idx]))
                    new_inst_config['exposure_count']= '1'
                    new_config['instrument_configs'].append(new_inst_config)
            # 4. Append this completed target configuration back to the main payload array
          data['requests'][0]['configurations'].append(new_config)
    return data

In [14]:
LCO_TOKEN ='TOKEN_HERE'
requestpath  = "https://observe.lco.global/api/requestgroups/"
lco_sent,lco_failed = [],[]
variables_all = []

PROPOSAL_ID  = '2013A-9999'
WINDOW_START = '2026-11-05 17:32:00'
WINDOW_END   = '2026-11-06 17:32:00'


# for json_df in list_jsons.tolist()[:]:
for json_df in [url_json]:

  target_list = make_targetlist(json_df)
  target_list = target_list[:-8]
  # change this if you do not want to submit the request, and just see how the payload looks
  send = True

  engeneering = True
  if engeneering:
    PROPOSAL_ID = '2013A-9999'

  # making sending the json file to LCO
  variables = {
        'ipp_value':1.05, #default
        'json_file_name': Path(url_json).stem,
        "PROPOSAL_ID": PROPOSAL_ID,
        'maximum_airmass': 2,
        'minimum_lunar_distance': 20,
        'windows': [{'start': WINDOW_START, 'end': WINDOW_END}],
        'targets': target_list[:-2]
    }


  variables_all.append(variables)

  #if IPP is left, you can bump up your request, use with caution
  if variables['json_file_name'] == '2_Shadow_Fornax_g90_2of3' or variables['json_file_name'] == '5_Shadow_Fornax_r90_2of3' or variables['json_file_name'] == '2026acxa':
    variables['ipp_value'] = 2

  data = make_payload(variables,template_path = template_json_url)

  if send:
    response = requests.post(
            requestpath,
            headers={"Authorization": f"Token {LCO_TOKEN}"},
            json=data,  # Make sure you use json!
        )

    if response.status_code == 400:
        print(variables['json_file_name'], 'Failed (sending to queue)', target_list[-2]['json_file_name'])
        lco_failed.append([variables['json_file_name'],response.text])


    elif response.status_code == 201 or response.status_code == 200:
        lco_sent.append([variables['json_file_name'],'sent!',response.json()['id']])
        print(variables['json_file_name'], 'Successfully sent')



example_json_coordinates Successfully sent


In [8]:
print('sources sent:', lco_sent)


sources sent: []


In [ ]:
print('sources failed:',lco_failed)

sources failed: [['example_json_coordinates', '{"requests":[{"non_field_errors":["According to the constraints of the request, the target will only be visible for a maximum of 0.41 hours within the time window. This is less than the duration of your request 0.62 hours. Consider expanding the time window or loosening the airmass or lunar separation constraints."]}]}']]


In [15]:
ids = np.asarray(lco_sent).T[-1]
print(ids)
for idlco in ids:
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    if response.status_code == 200:
      print('request:',idlco,'cancelled')

['2733094']
request: 2733094 cancelled
